<a href="https://colab.research.google.com/github/zwimpee/cursivetransformer/blob/main/debug_hct.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Clone the cursivetransformer repository and install its requirements
!rm -rf cursivetransformer && git clone https://github.com/zwimpee/cursivetransformer.git
!pip install -r cursivetransformer/requirements.txt

Cloning into 'cursivetransformer'...
remote: Enumerating objects: 2988, done.
remote: Counting objects: 100% (602/602), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 2988 (delta 553), reused 491 (delta 473), pack-reused 2386 (from 1)
Receiving objects: 100% (2988/2988), 64.41 MiB | 12.58 MiB/s, done.
Resolving deltas: 100% (1667/1667), done.
  Cloning https://github.com/callummcdougall/CircuitsVis.git to /tmp/pip-req-build-vho8hjjf
  Running command git clone --filter=blob:none --quiet https://github.com/callummcdougall/CircuitsVis.git /tmp/pip-req-build-vho8hjjf
  Resolved https://github.com/callummcdougall/CircuitsVis.git to commit 1e6129d08cae7af9242d9ab5d3ed322dd44b4dd3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 3.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of multiprocess to determine which 

In [2]:
import sys
sys.path.append('/content/cursivetransformer')
from cursivetransformer.model import get_all_args, get_checkpoint, get_latest_checkpoint_artifact
from cursivetransformer.data import create_datasets, offsets_to_strokes, strokes_to_offsets
from cursivetransformer.sample import generate, generate_n_words, plot_strokes
from cursivetransformer.mech_interp import (
    HookedCursiveTransformer,
    HookedCursiveTransformerConfig,
    convert_cursivetransformer_model_config,
    visualize_attention,
    generate_repeated_stroke_tokens,
    generate_random_ascii_context,
    run_and_cache_model_repeated_tokens,
    compute_induction_scores,
    plot_induction_scores,
    plot_head_attention_pattern,
    create_induction_summary,
    ablate_heads,
    get_induction_positions,
    compute_loss_on_induction_positions,
    visualize_attention_patterns
)

import pandas as pd
import os

import copy
import types
from typing import List, Callable, Dict, Optional, Union, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import einops
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio
import circuitsvis as cv
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from jaxtyping import Float, Int


import transformer_lens.utils as utils
from transformer_lens.hook_points import HookPoint
from transformer_lens import ActivationCache

torch.set_grad_enabled(False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# wandb_api_key - e56bbe426145e5983e72a933299daca195ebb6a7
args = get_all_args(False)
args.sample_only = True
args.max_seq_length = 1250
args.load_from_run_id = '7e9hz1og'
args.wandb_project = 'bigbank_2k'
args.wandb_entity = 'sam-greydanus'
args.dataset_name = 'bigbank'
args.wandb_run_name = 'cursivetransformer_mech_interp_part_3_stroke_attention'
torch.manual_seed(args.seed)
torch.cuda.manual_seed_all(args.seed)
train_dataset, test_dataset = create_datasets(args)
args.vocab_size = train_dataset.get_vocab_size()
args.block_size = train_dataset.get_stroke_seq_length()
args.context_block_size = train_dataset.get_text_seq_length()
args.context_vocab_size = train_dataset.get_char_vocab_size()

Enter your W&B API key: ··········
Trying to load dataset file from /content/cursivetransformer/data/bigbank.json.zip
Succeeded in loading the bigbank dataset; contains 2000 items.
For a dataset of 1900 examples we can generate 205257574037880 combinations of 5 examples.
Generating 497000 random combinations.
For a dataset of 100 examples we can generate 75287520 combinations of 5 examples.
Generating 3000 random combinations.
Number of examples in the train dataset: 497000
Number of examples in the test dataset: 3000
Max token sequence length: 1250
Number of unique characters in the ascii vocabulary: 71
Ascii vocabulary:
	" enaitoshrdx.vpukbgfcymzw1lqj804I92637OTAS5N)EHR"'(BCQLMWYU,ZF!DXV?KPGJ"
Split up the dataset into 497000 training examples and 3000 test examples


## Verify that the original and hooked versions of the model produce the same output logits for a given sequence

In [4]:
# Load the original model
original_model, _, _, _, _ = get_checkpoint(args, sample_only=True)
original_model.eval()

Number of Transformer parameters: 379392
Model #params: 403840
Finding latest checkpoint for W&B run id 7e9hz1og
  model:best_checkpoint:v129
  model:best_checkpoint:v130
  model:best_checkpoint:v131
  model:best_checkpoint:v132
  model:best_checkpoint:v133
  model:best_checkpoint:v134
  model:best_checkpoint:v135
  model:best_checkpoint:v136
  model:best_checkpoint:v137
  model:best_checkpoint:v138
  model:best_checkpoint:v139
  model:best_checkpoint:v140
  model:best_checkpoint:v141
  model:best_checkpoint:v142
  model:best_checkpoint:v143
  model:best_checkpoint:v144
  model:best_checkpoint:v145
  model:best_checkpoint:v146
  model:best_checkpoint:v147
  model:best_checkpoint:v148
  wandb-history:run-7e9hz1og-history:v4
Selected:  model:best_checkpoint:v148


wandb: Using wandb-core as the SDK backend. Please refer to https://wandb.me/wandb-core for more information.
wandb:   1 of 1 files downloaded.  


Transformer(
  (transformer): ModuleDict(
    (wte): Embedding(382, 64)
    (wpe): Embedding(1250, 64)
    (wce): Embedding(72, 64)
    (wcpe): Embedding(50, 64)
    (h): ModuleList(
      (0-3): 4 x Block(
        (ln_1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=64, out_features=192, bias=True)
          (c_proj): Linear(in_features=64, out_features=64, bias=True)
        )
        (ln_2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (cross_attn): CrossAttention(
          (c_attn_q): Linear(in_features=64, out_features=64, bias=True)
          (c_attn_kv): Linear(in_features=64, out_features=128, bias=True)
          (c_proj): Linear(in_features=64, out_features=64, bias=True)
        )
        (ln_3): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (mlp): ModuleDict(
          (c_fc): Linear(in_features=64, out_features=256, bias=True)
          (c_proj): Linear(in_fe

In [5]:
# Load the hooked model
cfg = convert_cursivetransformer_model_config(args)
hooked_model = HookedCursiveTransformer.from_pretrained("cursivetransformer", cfg)
hooked_model = hooked_model.to(device)
hooked_model.eval()

Loading pretrained model cursivetransformer
Finding latest checkpoint for W&B run id 7e9hz1og
  model:best_checkpoint:v129
  model:best_checkpoint:v130
  model:best_checkpoint:v131
  model:best_checkpoint:v132
  model:best_checkpoint:v133
  model:best_checkpoint:v134
  model:best_checkpoint:v135
  model:best_checkpoint:v136
  model:best_checkpoint:v137
  model:best_checkpoint:v138
  model:best_checkpoint:v139
  model:best_checkpoint:v140
  model:best_checkpoint:v141
  model:best_checkpoint:v142
  model:best_checkpoint:v143
  model:best_checkpoint:v144
  model:best_checkpoint:v145
  model:best_checkpoint:v146
  model:best_checkpoint:v147
  model:best_checkpoint:v148
  wandb-history:run-7e9hz1og-history:v4
Selected:  model:best_checkpoint:v148


wandb:   1 of 1 files downloaded.  


Original state dict keys: odict_keys(['transformer.wte.weight', 'transformer.wpe.weight', 'transformer.wce.weight', 'transformer.wcpe.weight', 'transformer.h.0.ln_1.weight', 'transformer.h.0.ln_1.bias', 'transformer.h.0.attn.bias', 'transformer.h.0.attn.c_attn.weight', 'transformer.h.0.attn.c_attn.bias', 'transformer.h.0.attn.c_proj.weight', 'transformer.h.0.attn.c_proj.bias', 'transformer.h.0.ln_2.weight', 'transformer.h.0.ln_2.bias', 'transformer.h.0.cross_attn.c_attn_q.weight', 'transformer.h.0.cross_attn.c_attn_q.bias', 'transformer.h.0.cross_attn.c_attn_kv.weight', 'transformer.h.0.cross_attn.c_attn_kv.bias', 'transformer.h.0.cross_attn.c_proj.weight', 'transformer.h.0.cross_attn.c_proj.bias', 'transformer.h.0.ln_3.weight', 'transformer.h.0.ln_3.bias', 'transformer.h.0.mlp.c_fc.weight', 'transformer.h.0.mlp.c_fc.bias', 'transformer.h.0.mlp.c_proj.weight', 'transformer.h.0.mlp.c_proj.bias', 'transformer.h.1.ln_1.weight', 'transformer.h.1.ln_1.bias', 'transformer.h.1.attn.bias', 'tr

HookedCursiveTransformer(
  (embed): Embedding(382, 64)
  (hook_embed): HookPoint()
  (pos_embed): Embedding(1250, 64)
  (hook_pos_embed): HookPoint()
  (blocks): ModuleList(
    (0-3): 4 x TransformerBlock(
      (ln1): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln3): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
      )
      (cross_attn): CrossAttention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pat

In [6]:
# Prepare a sample input
sample_tokens = torch.randint(0, args.vocab_size, (1, args.block_size)).to(device)
sample_context = torch.randint(0, args.context_vocab_size, (1, args.context_block_size)).to(device)

In [7]:
# Run both models
with torch.no_grad():
    original_output = original_model(sample_tokens, sample_context)
    hooked_output = hooked_model(sample_tokens, sample_context)

In [8]:
# Compare outputs
max_diff = torch.max(torch.abs(original_output[0] - hooked_output))
print(f"Maximum difference in logits: {max_diff.item()}")

if max_diff > 1e-5:
    print("Models are not equivalent!")
    # If models are not equivalent, let's print more detailed information
    print("Original model output shape:", original_output[0].shape)
    print("Hooked model output shape:", hooked_output.shape)
    print("Original model output mean:", original_output[0].mean().item())
    print("Hooked model output mean:", hooked_output.mean().item())
    print("Original model output std:", original_output[0].std().item())
    print("Hooked model output std:", hooked_output.std().item())
else:
    print("Models are equivalent.")

Maximum difference in logits: 99.28694152832031
Models are not equivalent!
Original model output shape: torch.Size([1, 1250, 382])
Hooked model output shape: torch.Size([1, 1250, 382])
Original model output mean: -7.335381507873535
Hooked model output mean: 0.011193220503628254
Original model output std: 11.664806365966797
Hooked model output std: 0.5616009831428528
